In [1]:
import sys, os
import numpy as np
import pandas as pd

sys.path.insert(0, "..")

OUT = "../data/processed/powerbi/"
os.makedirs(OUT, exist_ok=True)

ensemble  = pd.read_parquet("../artifacts/ensemble_test_predictions.parquet")
baselines = pd.read_parquet("../artifacts/baseline_test_predictions.parquet")
quantile  = pd.read_parquet("../artifacts/quantile_test_predictions.parquet")
outl      = pd.read_parquet("../artifacts/outl_results.parquet")
sens      = pd.read_csv("../data/processed/sensitivity_results.csv")
ids       = pd.read_csv("../data/processed/subsample_series_ids.csv")

In [2]:
forecasts = (
    ensemble[["id", "date", "store_id", "state_id", "cat_id", "dept_id",
              "sales", "pred_ensemble_trees"]]
    .merge(baselines[["id", "date", "Croston_SBA"]], on=["id", "date"], how="left")
    .merge(quantile[["id", "date", "q50", "q90", "q95", "q99"]], on=["id", "date"], how="left")
    .rename(columns={
        "sales": "actual",
        "pred_ensemble_trees": "forecast_ml",
        "Croston_SBA": "forecast_classical",
    })
)
cat_map   = {0: "Foods", 1: "Hobbies", 2: "Household"}
state_map = {0: "California", 1: "Texas", 2: "Wisconsin"}
forecasts["category"] = forecasts["cat_id"].map(cat_map)
forecasts["state"]    = forecasts["state_id"].map(state_map)
forecasts.to_csv(OUT + "forecasts.csv", index=False)
print(f"forecasts.csv: {forecasts.shape}")
forecasts.head()

forecasts.csv: (27610, 15)


,id,date,store_id,state_id,cat_id,dept_id,actual,forecast_ml,forecast_classical,q50,q90,q95,q99,category,state
0,FOODS_1_002_TX_2_validation,2016-03-01,5,1,0,0,0,0.147692,0.156544,0.000000e+00,1.002531,1.131836,2.431898,Foods,Texas
1,FOODS_1_002_TX_2_validation,2016-03-02,5,1,0,0,0,0.181619,0.156544,0.000000e+00,0.983034,1.145856,2.668314,Foods,Texas
2,FOODS_1_002_TX_2_validation,2016-03-03,5,1,0,0,0,0.156627,0.156544,0.000000e+00,1.007162,1.143769,2.524172,Foods,Texas
3,FOODS_1_002_TX_2_validation,2016-03-04,5,1,0,0,1,0.200543,0.156544,0.000000e+00,0.983034,1.151133,2.668314,Foods,Texas
4,FOODS_1_002_TX_2_validation,2016-03-05,5,1,0,0,0,0.213984,0.156544,7.575968e-08,1.033774,1.130132,2.648832,Foods,Texas


In [3]:
inventory = (
    outl.merge(ids, on="id", how="left")
)
inventory["category"] = inventory["cat_id"].map(cat_map)
inventory["state"]    = inventory["state_id"].map(state_map)
inventory["lead_time_label"] = inventory["lead_time"].astype(str) + " days"
inventory["service_level_label"] = (inventory["service_level"] * 100).astype(int).astype(str) + "%"

inventory.to_csv(OUT + "inventory_params.csv", index=False)
print(f"inventory_params.csv: {inventory.shape}")
inventory.head()

inventory_params.csv: (18036, 21)


,id,lead_time,service_level,policy,safety_stock,reorder_point,order_up_to_level,order_quantity,safety_stock_cost,cycle_stock_cost,...,ordering_cost,total_cost,cat_id,state_id,store_id,dept_id,category,state,lead_time_label,service_level_label
0,FOODS_1_002_TX_2_validation,7,0.90,Classical,1.598172,3.789783,69.019845,65.230061,2.146131,43.797633,...,43.797633,89.741397,0,1,5,0,Foods,Texas,7 days,90%
1,FOODS_1_002_TX_2_validation,7,0.90,ML_Gaussian,1.646404,4.127293,73.528925,69.401631,2.210901,46.598563,...,46.598563,95.408028,0,1,5,0,Foods,Texas,7 days,90%
2,FOODS_1_002_TX_2_validation,7,0.90,ML_Empirical_Quantile,3.504759,5.985648,75.387279,69.401631,4.706423,46.598563,...,46.598563,97.903550,0,1,5,0,Foods,Texas,7 days,90%
3,FOODS_1_002_TX_2_validation,7,0.95,Classical,2.051212,4.242823,69.472885,65.230061,2.754503,43.797633,...,43.797633,90.349769,0,1,5,0,Foods,Texas,7 days,95%
4,FOODS_1_002_TX_2_validation,7,0.95,ML_Gaussian,2.113117,4.594006,73.995637,69.401631,2.837634,46.598563,...,46.598563,96.034761,0,1,5,0,Foods,Texas,7 days,95%


In [4]:
# Aggregate cost by L, SL, policy — for executive-style charts
cost = (
    outl.groupby(["lead_time", "service_level", "policy"], as_index=False)
        .agg(
            total_safety_stock_units=("safety_stock", "sum"),
            total_safety_stock_cost =("safety_stock_cost", "sum"),
            total_holding_cost      =("holding_cost", "sum"),
            total_ordering_cost     =("ordering_cost", "sum"),
            total_cost              =("total_cost", "sum"),
            num_series              =("id", "nunique"),
        )
)
cost["service_level_label"] = (cost["service_level"] * 100).astype(int).astype(str) + "%"
cost.to_csv(OUT + "cost_analysis.csv", index=False)
print(f"cost_analysis.csv: {cost.shape}")
cost.head(12)

cost_analysis.csv: (36, 10)


,lead_time,service_level,policy,total_safety_stock_units,total_safety_stock_cost,total_holding_cost,total_ordering_cost,total_cost,num_series,service_level_label
0,7,0.90,Classical,2938.769864,1617.303607,33184.594600,31567.290993,64751.885593,501,90%
1,7,0.90,ML_Empirical_Quantile,3685.727188,2076.484567,33638.896376,31562.336499,65201.232874,501,90%
2,7,0.90,ML_Gaussian,2908.468822,1609.496347,33171.908156,31562.336499,64734.244654,501,90%
3,7,0.95,Classical,3771.834074,2075.766779,33643.057772,31567.290993,65210.348765,501,95%
4,7,0.95,ML_Empirical_Quantile,5132.421598,2879.464916,34441.876725,31562.336499,66004.213224,501,95%
5,7,0.95,ML_Gaussian,3732.943481,2065.746365,33628.158173,31562.336499,65190.494672,501,95%
6,7,0.99,Classical,5334.316740,2935.653387,34502.944380,31567.290993,66070.235373,501,99%
7,7,0.99,ML_Empirical_Quantile,8935.878906,5042.955318,36605.367127,31562.336499,68167.703626,501,99%
8,7,0.99,ML_Gaussian,5279.315716,2921.482016,34483.893825,31562.336499,66046.230324,501,99%
9,10,0.90,Classical,3238.366666,1782.181774,33349.472767,31567.290993,64916.763760,501,90%


In [5]:
sensitivity = sens.copy()
sensitivity["service_level_label"] = (sensitivity["service_level"] * 100).astype(int).astype(str) + "%"
sensitivity["stockout_mult_label"] = sensitivity["stockout_mult"].map({
    0.4: "Lost margin (0.4x)",
    1.0: "Full cost (1.0x)",
    2.0: "With reputation (2.0x)",
})
sensitivity.to_csv(OUT + "sensitivity.csv", index=False)
print(f"sensitivity.csv: {sensitivity.shape}")
sensitivity.head()

sensitivity.csv: (468, 11)


,mape,lead_time,service_level,stockout_mult,ss_gauss,ss_emp,tc_gauss,tc_emp,pct_reduction_TC,service_level_label,stockout_mult_label
0,3,7,0.90,0.4,93.466658,127.114655,83423.850946,82335.670079,1.304400,90%,Lost margin (0.4x)
1,3,7,0.90,1.0,93.466658,127.114655,113254.450355,110509.013964,2.424131,90%,Full cost (1.0x)
2,3,7,0.90,2.0,93.466658,127.114655,162972.116035,157464.587107,3.379430,90%,With reputation (2.0x)
3,3,7,0.95,0.4,119.962005,163.148327,85922.849733,72962.392534,15.083831,95%,Lost margin (0.4x)
4,3,7,0.95,1.0,119.962005,163.148327,119482.274068,87049.064477,27.144788,95%,Full cost (1.0x)


In [6]:
ml_results = pd.read_csv("../data/processed/all_model_results.csv")
best_classical = ml_results[~ml_results["model"].str.contains("LightGBM|XGBoost|RandomForest|LSTM|STACK")].sort_values("MAE").iloc[0]
best_ensemble  = ml_results[ml_results["model"] == "STACK_Ridge_trees"].iloc[0]

# Pull headline cost reduction from cost_analysis at 95% SL, L=14
pivot95 = cost[(cost["service_level"] == 0.95) & (cost["lead_time"] == 14)]
classical_tc = float(pivot95[pivot95["policy"] == "Classical"]["total_cost"].iloc[0])
ml_emp_tc    = float(pivot95[pivot95["policy"] == "ML_Empirical_Quantile"]["total_cost"].iloc[0])
cost_reduction = (classical_tc - ml_emp_tc) / classical_tc * 100

kpi = pd.DataFrame([
    {"kpi": "Best classical MAE",  "value": f"{best_classical['MAE']:.3f}",       "model": best_classical["model"]},
    {"kpi": "Best ensemble MAE",   "value": f"{best_ensemble['MAE']:.3f}",        "model": best_ensemble["model"]},
    {"kpi": "MAE improvement %",   "value": f"{(best_classical['MAE'] - best_ensemble['MAE'])/best_classical['MAE']*100:.1f}%", "model": "ensemble vs classical"},
    {"kpi": "Inventory cost reduction %", "value": f"{cost_reduction:.1f}%",      "model": "L=14, SL=95%, stockout 1.0x"},
    {"kpi": "Number of SKU-store series", "value": f"{ensemble['id'].nunique():,}", "model": "stratified subsample"},
    {"kpi": "Test window",         "value": f"{ensemble['date'].min().strftime('%Y-%m-%d')} to {ensemble['date'].max().strftime('%Y-%m-%d')}", "model": ""},
])
kpi.to_csv(OUT + "kpi_summary.csv", index=False)
display(kpi)

,kpi,value,model
0,Best classical MAE,0.970,SimpleAvg_trees
1,Best ensemble MAE,0.952,STACK_Ridge_trees
2,MAE improvement %,1.8%,ensemble vs classical
3,Inventory cost reduction %,-1.5%,"L=14, SL=95%, stockout 1.0x"
4,Number of SKU-store series,502,stratified subsample
5,Test window,2016-03-01 to 2016-04-24,
